# AmbiQT: Full 8,192-token three-agent experiment

**Purpose:** Recreate Sanjana's existing A / A2 / B pilot, changing **only the shared Qwen3:4B generation budget** from 4,096 to **8,192** tokens. The pilot has 12 AmbiQT questions × 3 agents = **36 generation requests**. Outputs and metadata are saved to Google Drive.

**You need:** (1) this notebook and (2) `ambiqt_colab_pilot.zip` from your original project. Use **Runtime → Change runtime type → T4 GPU** before beginning.

Run each cell in order. Do not select **Run all**: setup prompts for file upload and Drive permissions. Original agent prompts and evaluation logic stay in the supplied ZIP. **Do not upload the earlier saved results as a replacement for the ZIP.**

**Interpretation note:** 36 recorded requests ≠ 36 successful SQL outputs. We check `generation_status`, executable SQL, gold-result matches, and dual-interpretation recovery separately. This is a **12-question pilot**, not an untouched validation holdout.

## 1. Upload the original ZIP and extract its full directory structure

In [1]:
from google.colab import files
from pathlib import Path
import zipfile, os, shutil

zip_path = Path('/content/ambiqt_colab_pilot.zip')
if not zip_path.exists():
    print('Upload ambiqt_colab_pilot.zip now:')
    uploaded = files.upload()
    if 'ambiqt_colab_pilot.zip' not in uploaded:
        raise FileNotFoundError('Please upload the file named ambiqt_colab_pilot.zip')

project = Path('/content/colab_pilot')
if project.exists():
    print('Existing /content/colab_pilot found; retaining it. To reset, restart runtime first.')
else:
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content')

assert (project / 'run_pilot.py').exists(), 'Missing run_pilot.py — check ZIP structure'
assert (project / 'option_c/agents/config.py').exists(), 'Missing agent config — check ZIP'
os.chdir(project)
print('Project root:', os.getcwd())
print('Scripts:', [p.name for p in project.glob('*.py')])

Upload ambiqt_colab_pilot.zip now:


Saving ambiqt_colab_pilot.zip to ambiqt_colab_pilot.zip
Project root: /content/colab_pilot
Scripts: ['run_pilot.py', 'prepare_and_verify.py', 'test_orchestration.py', 'test_run_pilot_config.py', 'run_reasoning_probe.py', 'run_budget_diagnostic.py', 'test_reasoning_probe.py', 'ollama_health.py', 'test_budget_diagnostic.py', 'smoke_test.py']


## 2. Install dependencies and verify the 12-record benchmark bundle

In [3]:
%cd /content/colab_pilot
!pip -q install -r requirements.txt
!python prepare_and_verify.py

/content
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.31.0 which is incompatible.
[1] imports
  PASS all required modules import
[2] selected DB paths resolve
  PASS manifest has 12 ELIGIBLE records (12)
  PASS DB resolves: col-synonyms#328 cre_Doc_Template_Mgt__8144fc9f45333fdf.sqlite
  PASS DB resolves: col-synonyms#1239 real_estate_properties__33d5f3dbfad77405.sqlite
  PASS DB resolves: col-synonyms#446 museum_visit__bce7d418f5b797b2.sqlite
  PASS DB resolves: tbl-synonyms#538 cre_Doc_Template_Mgt__1ec2ac4f42fffb1c.sqlite
  PASS DB resolves: tbl-synonyms#1415 real_estate_properties__3bd540543e10530a.sqlite
  PASS DB resolves: tbl-synonyms#613 museum_visit__7ef837978869a210.sqlite
  PASS DB resolves: tbl-split#73 cre_Doc_Template_Mgt__2badf7fc65c991df.sqlite
  PASS DB resolves: tbl

Expect **`RESULT: ALL CHECKS PASS`**. If this fails, stop here and inspect the traceback before running model inference.

## 3. Install Ollama (including missing `zstd`)

In [4]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama
!ollama --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama


## 4. Start Ollama and pull the exact model `qwen3:4b`

In [5]:
import subprocess, time, requests

url = 'http://127.0.0.1:11434/api/tags'
try:
    alive = requests.get(url, timeout=3).ok
except requests.RequestException:
    alive = False

if not alive:
    ollama_proc = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=open('/content/ollama_server.log', 'a'),
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    print('Started Ollama PID:', ollama_proc.pid)
else:
    print('Ollama already responding; no new server started.')

for attempt in range(45):
    try:
        if requests.get(url, timeout=3).ok:
            print('Ollama server ready')
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Ollama never became ready; inspect /content/ollama_server.log')

!ollama pull qwen3:4b
!ollama list

Started Ollama PID: 2269
Ollama server ready

NAME        ID              SIZE      MODIFIED               
qwen3:4b    359d7dd4bcda    2.5 GB    Less than a second ago    


## 5. Confirm GPU and run the longer smoke test

In [6]:
!nvidia-smi
%cd /content/colab_pilot
!python smoke_test.py --smoke-timeout 180

Tue Nov 24 19:36:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Required:** `RESULT: SMOKE TEST PASS`. The 180-second smoke timeout is deliberately longer than the original notebook's default, which previously timed out during model warmup. A `length` result signals token exhaustion, not necessarily a networking failure.

## 6. Mount Google Drive and preserve both runs

In [7]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
OUT = Path('/content/drive/MyDrive/ambiqt_pilot')
OUT.mkdir(parents=True, exist_ok=True)
print('Saving all generated JSONL and metadata under:', OUT)
print('Existing results:', [p.name for p in sorted(OUT.glob('pilot_generations_*.jsonl'))])

Mounted at /content/drive
Saving all generated JSONL and metadata under: /content/drive/MyDrive/ambiqt_pilot
Existing results: ['pilot_generations_20261124T191615Z.jsonl']


## 7. Verify the shared 8,192-token configuration; extend ONLY the runner preflight timeout

In [8]:
import sys
from pathlib import Path
sys.path.insert(0, '/content/colab_pilot')
from run_pilot import pilot_config

cfg = pilot_config(num_predict=8192)
print(cfg.generation_settings())
assert cfg.model == 'qwen3:4b'
assert cfg.num_predict == 8192
assert cfg.temperature == 0.0
assert cfg.think is True, 'Use original think=True setting for comparison with 4,096-token pilot'

# The runner has a separate, hard-coded 60s smoke timeout.
# Raise ONLY that preflight timeout to 180s; no agent prompts,
# generation settings, record selection, or scoring are changed.
runner = Path('/content/colab_pilot/run_pilot.py')
source = runner.read_text()
old = 'H.preflight(cfg, verbose=True)'
new = 'H.preflight(cfg, smoke_timeout=180, verbose=True)'
if old in source:
    (runner.parent / 'run_pilot_unmodified_backup.py').write_text(source)
    runner.write_text(source.replace(old, new, 1))
    print('Preflight smoke timeout extended to 180s; backup saved.')
elif new in source:
    print('Preflight timeout already extended.')
else:
    raise RuntimeError('Unexpected runner version; check run_pilot.py before proceeding')

{'model': 'qwen3:4b', 'temperature': 0.0, 'top_p': 1.0, 'num_predict': 8192, 'think': True, 'request_timeout': 600.0}
Preflight smoke timeout extended to 180s; backup saved.


## 8. Launch all 36 requests with `num_predict=8192`

**This is the full run.** It generates new timestamped result files on Drive and does not overwrite your old 4,096-token results. It can take substantial time on a T4 GPU. Leave the session connected. If it disconnects, any rows that were already flushed to Drive may still be recoverable.

In [1]:
!nvidia-smi


Tue Nov 24 19:34:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
%cd /content/colab_pilot

import sys
import json
from dataclasses import replace
from pathlib import Path

sys.path.insert(0, "/content/colab_pilot/option_c/agents")

import agent_a
from config import DEFAULT_CONFIG

path = Path(
    "/content/colab_pilot/AmbiQT-master/"
    "AmbiQT-master/benchmark/col-synonyms/validation.json"
)

with open(path, encoding="utf-8") as f:
    records = json.load(f)

record = records[328]

cfg = replace(
    DEFAULT_CONFIG,
    num_predict=8192,
    request_timeout=180
)

print("Running Agent A on question #328...", flush=True)

result = agent_a.generate(
    record["question"],
    record["schema_without_content"],
    config=cfg
)

print("Status:", result.generation_status)
print("Done reason:", result.done_reason)
print("Latency:", round(result.latency, 1))
print("Tokens:", result.total_generated_tokens)
print("Candidates:", list(result.candidates))
print("Error:", result.generation_error)

/content
Running Agent A on question #328...
Status: request_error
Done reason: None
Latency: 180.1
Tokens: None
Candidates: []
Error: TimeoutError: timed out


In [3]:
%cd /content/colab_pilot
!python -u run_pilot.py --out '/content/drive/MyDrive/ambiqt_pilot' --num-predict 8192

/content
preflight: Ollama readiness + model + smoke (num_predict=8192) ...
Traceback (most recent call last):
  File "/content/colab_pilot/run_pilot.py", line 181, in <module>
    run(a.out, timeout=a.timeout, error_threshold=a.error_threshold,
  File "/content/colab_pilot/run_pilot.py", line 72, in run
    ok, detail = H.preflight(cfg, verbose=True)
  File "/content/colab_pilot/ollama_health.py", line 157, in preflight
    ready, info = check_ready(config, timeout=ready_timeout)
  File "/content/colab_pilot/ollama_health.py", line 87, in check_ready
    sleep(interval)
KeyboardInterrupt
^C


Expected end of successful orchestration: `RUN OK: 36/36 requests written, 0 request_error (num_predict=8192)`. **That is not a guarantee of 36 complete generations.**

## 9. Find and summarize the new 8,192-token run (robust to prior files)

In [10]:
import json
import pandas as pd
from pathlib import Path

results = []
for meta_path in sorted(OUT.glob('pilot_meta_*.json')):
    meta = json.loads(meta_path.read_text())
    generation_path = OUT / f"pilot_generations_{meta['run_id']}.jsonl"
    if not generation_path.exists():
        continue
    with generation_path.open(encoding='utf-8') as f:
        run_rows = [json.loads(line) for line in f if line.strip()]
    results.append((meta, generation_path, run_rows))

if not results:
    raise FileNotFoundError('No saved pilot runs found in Google Drive.')

print('Available saved runs:')
for meta, path, rr in results:
    print(meta.get('run_id'), 'budget=', meta.get('num_predict'),
          'rows=', len(rr), 'file=', path.name)

new_runs = [(m, p, r) for m, p, r in results if m.get('num_predict') == 8192]
if not new_runs:
    raise RuntimeError('No 8192-token results yet. Run Step 8 first, or check Google Drive.')
new_meta, new_path, new_rows = new_runs[-1]
assert new_meta['num_predict'] == 8192
print('\nSelected 8192 run:', new_path.name)

def stats_for(rr, label):
    summaries = []
    for agent in ('A', 'A2', 'B'):
        subset = [r for r in rr if r.get('agent') == agent]
        completed = sum(r.get('generation_status') == 'ok' and r.get('num_candidates',0) > 0 for r in subset)
        truncated = sum(r.get('done_reason') == 'length' or r.get('generation_status') == 'truncated' for r in subset)
        executable = sum(any(c.get('execution_status') == 'ok' for c in r.get('candidate_scores', [])) for r in subset)
        matched = sum(any(c.get('result_match') is True for c in r.get('candidate_scores', [])) for r in subset)
        both = sum((r.get('record_score') or {}).get('both_recovered') is True for r in subset)
        summaries.append({'run': label, 'agent': agent, 'requests': len(subset),
                          'completed': completed, 'truncated': truncated,
                          'executable': executable, 'gold_matches': matched,
                          'both_recovered': both})
    return summaries

new_summary = pd.DataFrame(stats_for(new_rows, '8192'))
display(new_summary)
new_summary.to_csv(OUT/'summary_8192.csv', index=False)
print('Saved:', OUT/'summary_8192.csv')

Available saved runs:
20261124T191615Z budget= 8192 rows= 1 file= pilot_generations_20261124T191615Z.jsonl
20261124T194002Z budget= 8192 rows= 36 file= pilot_generations_20261124T194002Z.jsonl

Selected 8192 run: pilot_generations_20261124T194002Z.jsonl


,run,agent,requests,completed,truncated,executable,gold_matches,both_recovered
0,8192,A,12,12,0,12,8,0
1,8192,A2,12,12,0,12,8,3
2,8192,B,12,11,0,11,7,4


Saved: /content/drive/MyDrive/ambiqt_pilot/summary_8192.csv


## 10. Compare 4,096 and 8,192 using the preserved run files

In [11]:
old_runs = [(m, p, r) for m, p, r in results if m.get('num_predict') == 4096]
if old_runs:
    old_meta, old_path, old_rows = old_runs[-1]
    comparison = pd.DataFrame(stats_for(old_rows, '4096') + stats_for(new_rows, '8192'))
    display(comparison)
    comparison.to_csv(OUT/'comparison_4096_vs_8192.csv', index=False)
    print('Saved:', OUT/'comparison_4096_vs_8192.csv')
    print('4096 source:', old_path.name)
    print('8192 source:', new_path.name)
else:
    print('The earlier 4096-token file was not found in this Drive folder.')
    print('Previously recorded aggregate totals (not independently recovered from Drive here):')
    display(pd.DataFrame([
      {'run':'4096 prior summary','agent':'A','requests':12,'completed':11,'truncated':1,'executable':11,'gold_matches':8,'both_recovered':0},
      {'run':'4096 prior summary','agent':'A2','requests':12,'completed':9,'truncated':3,'executable':9,'gold_matches':6,'both_recovered':2},
      {'run':'4096 prior summary','agent':'B','requests':12,'completed':8,'truncated':4,'executable':8,'gold_matches':5,'both_recovered':3},
    ]))

The earlier 4096-token file was not found in this Drive folder.
Previously recorded aggregate totals (not independently recovered from Drive here):


,run,agent,requests,completed,truncated,executable,gold_matches,both_recovered
0,4096 prior summary,A,12,11,1,11,8,0
1,4096 prior summary,A2,12,9,3,9,6,2
2,4096 prior summary,B,12,8,4,8,5,3


## 11. See which questions still truncated, and export a presentation-ready summary

In [12]:
failed = pd.DataFrame([
    {'agent':r.get('agent'), 'category':r.get('category'), 'index':r.get('index'),
     'status':r.get('generation_status'), 'done_reason':r.get('done_reason'),
     'generated_tokens':r.get('total_generated_tokens'),
     'error':r.get('generation_error')}
    for r in new_rows if r.get('generation_status') != 'ok'
])
if len(failed):
    display(failed)
    failed.to_csv(OUT/'failures_8192.csv', index=False)
    print('Saved:', OUT/'failures_8192.csv')
else:
    print('All 8192 generations reported ok.')

print('\nFiles to share with teammate:')
print('1.', new_path)
print('2.', OUT / f"pilot_meta_{new_meta['run_id']}.json")
print('3.', OUT / 'summary_8192.csv')
if old_runs:
    print('4.', OUT / 'comparison_4096_vs_8192.csv')

,agent,category,index,status,done_reason,generated_tokens,error
0,B,col-synonyms,328,parse_error,stop,5739,no valid interpretation/SQL pair parsed from r...


Saved: /content/drive/MyDrive/ambiqt_pilot/failures_8192.csv

Files to share with teammate:
1. /content/drive/MyDrive/ambiqt_pilot/pilot_generations_20261124T194002Z.jsonl
2. /content/drive/MyDrive/ambiqt_pilot/pilot_meta_20261124T194002Z.json
3. /content/drive/MyDrive/ambiqt_pilot/summary_8192.csv


## 12. Next steps after the run

1. **Check completion first**: if A2 or B still truncates, report failures rather than counting them as SQL mistakes; keep all 12 questions in each denominator.
2. **Judge the research question with two metrics**: *at least one gold-result match* and *both interpretations recovered*. More complete SQL does not necessarily mean more accurate SQL.
3. **Confirm methodological controls**: model, temperature, token budget, question/schema inputs, and evaluation protocol should match. Agent prompts necessarily differ in their core instructions; standardize their shared framing for a stronger ablation.
4. **Do not claim a benchmark holdout result**: this ZIP is a 12-question selected pilot from AmbiQT's named `validation.json` files. Complete your course's 70/20/10 split with the full eligible dataset separately, preserving a truly unseen final holdout.
5. **Finish the final deliverables**: public GitHub, partial/final paper novelty + results + limitations, unlisted recording, and Turnitin report. Document exact run IDs and configs.

**If the T4 runtime disconnects:** remount Google Drive in a fresh notebook and begin at Step 9 to inspect already-saved results. For new generations, reinstall the runtime dependencies and Ollama, then run Step 8.

In [13]:
import json

path = "/content/drive/MyDrive/ambiqt_pilot/pilot_generations_20261124T194002Z.jsonl"

with open(path, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print("8,192-token experiment results")
print("-" * 40)

for agent in ["A", "A2", "B"]:
    group = [r for r in rows if r["agent"] == agent]

    completed = sum(r["generation_status"] == "ok" for r in group)

    executable = sum(
        any(
            c.get("execution_status") == "ok"
            for c in r.get("candidate_scores", [])
        )
        for r in group
    )

    correct = sum(
        any(
            c.get("result_match") is True
            for c in r.get("candidate_scores", [])
        )
        for r in group
    )

    recovered = sum(
        bool((r.get("record_score") or {}).get("both_recovered"))
        for r in group
    )

    print(f"\nAgent {agent}")
    print(f"Completed: {completed}/12")
    print(f"Executable: {executable}/12")
    print(f"Gold-result matches: {correct}/12")
    print(f"Both interpretations recovered: {recovered}/12")

8,192-token experiment results
----------------------------------------

Agent A
Completed: 12/12
Executable: 12/12
Gold-result matches: 8/12
Both interpretations recovered: 0/12

Agent A2
Completed: 12/12
Executable: 12/12
Gold-result matches: 8/12
Both interpretations recovered: 3/12

Agent B
Completed: 11/12
Executable: 11/12
Gold-result matches: 7/12
Both interpretations recovered: 4/12


new section

In [7]:
from google.colab import drive

drive.mount('/content/mydrive')

Mounted at /content/mydrive


In [8]:
import json
from pathlib import Path

folder = Path("/content/mydrive/MyDrive/ambiqt_pilot")
results_file = folder / "pilot_generations_20261124T194002Z.jsonl"

with open(results_file, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print("Loaded", len(rows), "records")

Loaded 36 records


In [15]:
import json
from pathlib import Path

folder = Path("/content/mydrive/MyDrive/ambiqt_pilot")

results_file = folder / "pilot_generations_20261124T194002Z.jsonl"

with open(results_file, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print("Loaded", len(rows), "agent-question records")

failures = [
    r for r in rows
    if r["generation_status"] != "ok"
]

for r in failures:
    print("\n" + "=" * 60)
    print("Agent:", r["agent"])
    print("Question:", r["category"], "#", r["index"])
    print("Status:", r["generation_status"])
    print("Termination:", r["done_reason"])
    print("Tokens:", r["total_generated_tokens"])
    print("Error:", r["generation_error"])

    print("\nRAW MODEL RESPONSE:")
    print(repr(r.get("raw_response")))

Loaded 36 agent-question records

Agent: B
Question: col-synonyms # 328
Status: parse_error
Termination: stop
Tokens: 5739
Error: no valid interpretation/SQL pair parsed from response

RAW MODEL RESPONSE:
'The question asks for the **id and type code** of the **template used by the most documents**. Based on the schema provided, the correct SQL query to answer this question is as follows:\n\n- We join the `templates` table with the `documents` table on `templates.template_id = documents.template_id` to count how many documents are associated with each template.\n- We then join the `templates` table with the `ref_template_types` table on `templates.document_type_code = ref_template_types.template_type_code` to link each template to its corresponding type code.\n- We group by `templates.template_id`, order by the count of documents in descending order, and limit the result to the top 1.\n\nThis ensures we get the **template id** (from the `templates` table) and its **type code** (from th

In [16]:
import sys

sys.path.insert(
    0, "/content/colab_pilot/option_c/agents"
)

from agent_b import parse_interpretations

failure = next(
    r for r in rows
    if r["agent"] == "B"
    and r["category"] == "col-synonyms"
    and r["index"] == 328
)

parsed = parse_interpretations(
    failure["raw_response"]
)

print("Valid interpretation/SQL pairs:", len(parsed))
print(parsed)

Valid interpretation/SQL pairs: 0
[]
